# Path Configuration

In [37]:
from pathlib import Path
import pandas as pd
import json, re
import numpy as np

IN_CSV = Path("processed_dataset.csv")
OUT_XLSX = Path("processed_dataset2.xlsx")
OUT_XLSX_CHUNK_PREFIX = Path("processed_dataset2_chunk")
assert IN_CSV.exists(), f"File input tidak ditemukan: {IN_CSV}"
df = pd.read_csv(IN_CSV, dtype=object)
print("Loaded rows:", len(df))

Loaded rows: 25384


# Fungsi Deteksi dan Sanitize

In [38]:
# pattern control chars (C0 and C1)
_ctrl_re = re.compile(r'[\x00-\x1F\x7F-\x9F]')
# Excel max cell length (32,767) — pakai margin
EXCEL_MAX = 32760

def obj_to_str_safe(x):
    """Convert non-str objects (dict/list/tuple) to JSON string; keep None as None."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, str):
        return x
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return str(x)

def sanitize_text(s, maxlen=EXCEL_MAX):
    """Remove control chars, trim, collapse whitespace, truncate to maxlen."""
    if s is None:
        return None
    # ensure string
    s = str(s)
    # remove control chars
    s = _ctrl_re.sub(" ", s)
    # collapse repeated whitespace (including newlines) to single space but keep some structure
    s = re.sub(r'\s+', ' ', s).strip()
    # truncate if too long
    if len(s) > maxlen:
        s = s[:maxlen-3] + "..."
    return s

def sanitize_cell(x):
    # convert objects to string if needed
    if isinstance(x, (dict, list, tuple)):
        s = obj_to_str_safe(x)
    else:
        s = x
    # sanitize string-like
    if s is None:
        return None
    return sanitize_text(s)

# Preview Kolom/Sel Bermasalah

In [39]:
# Kolom yang mengandung non-str objects (dict/list)
cols_with_objects = []
for c in df.columns:
    # check a sample up to 200 rows
    sample = df[c].dropna().head(200)
    if any(isinstance(v, (dict, list, tuple)) for v in sample):
        cols_with_objects.append(c)
cols_with_objects

# Kolom yang punya control chars or very long strings (scan sample)
cols_with_ctrl = []
cols_with_long = []
for c in df.columns:
    found_ctrl = False
    found_long = False
    for v in df[c].dropna().head(500):  # check up to 500 non-null values per kolom
        if not isinstance(v, str):
            s = str(v)
        else:
            s = v
        if _ctrl_re.search(s):
            found_ctrl = True
        if len(s) > 10000:  # suspiciously long
            found_long = True
        if found_ctrl and found_long:
            break
    if found_ctrl:
        cols_with_ctrl.append(c)
    if found_long:
        cols_with_long.append(c)

print("Columns with object types (sample):", cols_with_objects)
print("Columns containing control chars (sample):", cols_with_ctrl)
print("Columns with very long strings (sample):", cols_with_long)

# contoh baris yang bermasalah (control chars) -- lihat 10 contoh
bad_rows = []
for idx, row in df.iterrows():
    for c in cols_with_ctrl:
        v = row.get(c)
        if pd.isna(v) or v is None:
            continue
        if _ctrl_re.search(str(v)):
            bad_rows.append((idx, c, str(v)[:200]))  # tampilkan preview
            break
    if len(bad_rows) >= 10:
        break

bad_rows

Columns with object types (sample): []
Columns containing control chars (sample): ['title', 'description', 'p_alamat']
Columns with very long strings (sample): []


[(0,
  'description',
  'Rumah 2lt 10x15 148m private pool type 3KT Cluster Victoria Metland Cakung\n  \n Rumah siap huni di cluster victoria\n Bangunan 2 lantai\n Luas tanah : 148m\n Dimensi : 10x14,8\n Luas bangunan : 200m\n Kamar'),
 (1,
  'description',
  '\xa0RUMAH jual di GRIYA HARAPAN PERMAI. Kawasan harapan indah Bekasi\n \n Luas Tanah : 180m²\n Luas Bangunan : 140m² ( 1,5 Lantai) HOOK\n Kamar Tidur : 4\n Kamar Mandi : 2\n Listrik : 2.200 W\n Sertifikat : SHM'),
 (2,
  'description',
  'Promo Samesta East Point Oktober 2024\n- Rp2 Juta bisa langsung akad!*\n- DP 0%*\n- Gratis PPN 100%*\n- Gratis Furnished dari Dekoruma*\n- Gratis Biaya Akad (Asuransi Kebakaran, Asuransi Jiwa)*\n*SK Berlaku'),
 (3,
  'description',
  'Dijual Murah\n Apt.WGP tower A lt.6 (Hook)\n Kelapa Gading \n Jakarta Utara\n \n Luas 40m2\n KT 1 (1 kmr dibongkar)\n KM 1 \n Full Furnish\n View City\n Sertifikat\n Hrg 460 jt nepis\n \n Hub: vjl\n Aifi - GPS'),
 (4,
  'description',
  'Jual Murah Rumah Siap Huni 

# Pembersihan Dataframe

In [40]:
# Buat salinan
df_clean = df.copy()

# Hapus kolom bantu internal kalau ada
for helper in ["_params_dict","_main_bed","_main_bath","_main_area","_raw_ad_preview"]:
    if helper in df_clean.columns:
        df_clean.drop(columns=[helper], inplace=True)

# Sanitize per kolom: untuk kolom object-like, convert & sanitize
for c in df_clean.columns:
    # apply conversion then sanitize
    # but done in a vectorized way for speed where possible
    # first convert dict/list -> JSON string for those entries
    def conv_and_sanitize(val):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return None
        if isinstance(val, (dict, list, tuple)):
            s = obj_to_str_safe(val)
        else:
            s = val
        return sanitize_text(s)
    # apply
    df_clean[c] = df_clean[c].apply(conv_and_sanitize)

# quick check
print("After sanitize non-null counts (top):")
print(df_clean.notnull().sum().sort_values(ascending=False).head(30))

After sanitize non-null counts (top):
ad_id                  25384
created_at             25384
main_info              25384
first_image_url        25384
images_count           25384
status                 25384
user_id                25384
lon                    25384
building_area_m2       25384
lat                    25384
city_name              25384
province_name          25384
country_name           25384
raw_parameters         25384
source_file            25384
bathrooms              25384
land_area_m2           25384
bedrooms               25384
title                  25383
user_name              25381
description            25376
property_type          25348
sublocality_name       25285
price_raw              25125
price_display          25125
currency               25125
external_source_url    22001
floor                  18877
p_alamat               12300
dtype: int64


# Deteksi Ulang Problem Setelah Sanitasi

In [41]:
# Check no more control chars & no dicts/lists
any_ctrl = False
any_obj = False
for c in df_clean.columns:
    sample = df_clean[c].dropna().head(200)
    for v in sample:
        if _ctrl_re.search(str(v)):
            any_ctrl = True
        if isinstance(v, (dict, list, tuple)):
            any_obj = True
    if any_ctrl or any_obj:
        break
print("Any control-chars remaining (sampled)?", any_ctrl)
print("Any object-type cells remaining (sampled)?", any_obj)

Any control-chars remaining (sampled)? False
Any object-type cells remaining (sampled)? False


# Simpan ke Excel

In [42]:
try:
    df_clean.to_excel(OUT_XLSX, index=False)
    print("Saved clean Excel:", OUT_XLSX)
except Exception as e:
    print("Direct Excel save failed:", repr(e))
    # Fallback: split into multiple smaller excel files (sheet-per-chunk or file-per-chunk)
    CHUNK = 5000
    n = len(df_clean)
    parts = (n + CHUNK -1)//CHUNK
    print(f"Attempting fallback: saving in {parts} parts, chunk={CHUNK}")
    for i in range(parts):
        start = i*CHUNK
        end = min((i+1)*CHUNK, n)
        outp = Path(f"{OUT_XLSX_CHUNK_PREFIX}_{i+1}.xlsx")
        try:
            df_clean.iloc[start:end].to_excel(outp, index=False)
            print("Saved part:", outp, "rows:", end-start)
        except Exception as e2:
            print("Failed saving part", i+1, "error:", repr(e2))
    print("Fallback done; check part files.")

Saved clean Excel: processed_dataset2.xlsx


# Simpan ke CSV / Compressed Excel
### (jika kode pada sel sebelumnya gagal)

In [ ]:
# As last resort, save CSV and a zipped CSV
CSV_OUT = OUT_XLSX.with_suffix(".csv")
df_clean.to_csv(CSV_OUT, index=False, encoding="utf-8")
print("Saved CSV fallback:", CSV_OUT)
# compress CSV
import gzip, shutil
with open(CSV_OUT, 'rb') as f_in, gzip.open(str(CSV_OUT)+'.gz','wb') as f_out:
    shutil.copyfileobj(f_in, f_out)
print("Saved compressed CSV:", str(CSV_OUT)+'.gz')

# Tunjukkan Nilai "floor" yang Anomali dan Contoh "p_alamat"

In [43]:
# Asumsi: df sudah ada di notebook (hasil sebelumnya)
import pandas as pd
import numpy as np
pd.options.display.max_colwidth = 200

# show distribution & anomalies for 'floor'
if "floor" in df.columns:
    df['floor_num'] = pd.to_numeric(df['floor'], errors='coerce')
    print("Floor stats (describe):")
    print(df['floor_num'].describe(percentiles=[0.5,0.75,0.9,0.99]))
    # show rows where floor is unexpectedly large (>=50) or negative
    suspicious = df[(df['floor_num'].notna()) & ((df['floor_num'] >= 50) | (df['floor_num'] < 0))]
    print("\nCount suspicious floor values (>=50 or <0):", len(suspicious))
    display(suspicious[['ad_id','title','floor','building_area_m2','main_info','description']].head(10))

# show strange p_alamat samples
if "p_alamat" in df.columns:
    print("\nSample p_alamat (some rows where p_alamat non-null but short/looks junk):")
    samp = df[df['p_alamat'].notnull()].sample(n=min(12, df['p_alamat'].notnull().sum()), random_state=42)
    display(samp[['ad_id','p_alamat','sublocality_name','city_name','province_name','description']].head(12))

Floor stats (describe):
count    18877.000000
mean        46.712666
std        201.632926
min          0.000000
50%          2.000000
75%         26.000000
90%        128.000000
99%        540.480000
max      14410.000000
Name: floor_num, dtype: float64

Count suspicious floor values (>=50 or <0): 4313


,ad_id,title,floor,building_area_m2,main_info,description
7,938511300,Dijual cepat rumah dalam cluster di Metland MentengJ Cakung Jak-Tim,96.0,124.0,3 KT - 2 KM - 124 m2,Dijual cepat rumah dalam cluster di Metland MentengJ Cakung Jakarta Timur\n\nLT 96m² (6 X 16)\n\nLB 124m²\n\nKT 3\n\nKM 2\n\n2¹/2 lantai\n\nPAM\n\nPLN 2200W\n\nSudah renovasi\n\nSHM\n\nHarga 2.5M ...
40,938354838,Dijual Rumah 2 Lantai Bagus Siap Huni Di Cluster Samata Harapan Indah,133.0,80.0,3 KT - 3 KM - 80 m2,"Turun Harga, \n\n\n\n\nDijual Rumah Bagus Siap Huni Di Cluster Samata Harapan Indah Bekasi\n\n\n\n\nLt 133m\n\nLb 80m\n\nKt 2 + 1\n\nKm 2 + 1\n\nListrik 2200w\n\nAir PAM\n\nSHM\n\nCarport 2 mbl\n..."
46,939121496,Dijual Rumah Mungil Cantik Bangunan Baru Harapan Indah 1 Bekasi,77.0,60.0,2 KT - 1 KM - 60 m2,Dijual Rumah Bangunan Baru\nJl Kedondong\nHarapan Indah 1\nBekasi\n\nLT: 77 m2\nLB: 60 m2\nKamar Tidur: 2\nKamar Mandi: 1\nDaya Listrik: 2200 Kwh\nAir: PDAM\nCarport: 1 mobil\nAkses jalan muat 2 m...
72,927178568,Jual Cepat Rumah Minimalis Jalan 2 Mobil di Kelapa Cengkir,90.0,1.0,3 KT - 2 KM - 1 m2,"JUAL CEPAT Rumah Minimalis Jalan 2 Mobil di Kelapa Cengkir, Harga OK, Bisa dibantu KPR sampai gooaall\n \n\n LT : 90 m2 (6x15)\n \n\n 2,5 Lantai\n KT : 3\n KM : 2\n PAM\n Kitchen Set\n Carpot 1 Mo..."
74,927144390,Jual Cepat Rumah Minimalis Tinggi Dari Jalan di Kelapa Molek,90.0,1.0,5 KT - 3 KM - 1 m2,"JUAL CEPAT! Rumah Minimalis Tinggi dari Jalan di Kelapa Molek, Bisa Nego, Bisa dibantu KPR sampai gooaall\n \n\n LT : 90 m2 (5x18)\n \n\n 3 Lantai\n KT : 5\n KM : 3\n PAM\n Kitchen Set\n Ada AC\n ..."
93,939945889,"Jual Rumah di Cluster Cassia JGC – Akses Mudah, Keamanan 24 Jam!",156.0,148.0,5 KT - 4 KM - 148 m2,"Rumah Minimalis Modern di cluster Cassia, JGC, Jaktim\n\nLT 156m2 (9x17m) \nLB 148m2\n2 lantai\nKT 4+1\nKM 2+2\nHadap selatan\nCarport 2 mobil\nListrik 3500w\nAir pam\n4 AC\nKitchen set\nSHM\n\nDe..."
102,931950583,S674 Rumah 167 m2 Dekat Taman di Pondok Kopi Jakarta Timur,167.0,200.0,4 KT - 3 KM - 200 m2,"DIJUAL RUMAH SECOND 1,5 LANTAI DI PERUMAHAN PONDOK KOPI JAKARTA TIMUR\n \n MASIH KOKOH DAN RAPI / DIRENOVASI TAHUN 2020\n \n\n HARGA 2,8 M NEGO\n SHM, IMB\n \n LT 167 m²\n LB 200 m²\n \n Kamar Tid..."
123,937759416,Rumah Minimalis Kelapa Gading Janur Elok Asri Hibrida Molek Kopyor,90.0,150.0,4 KT - 3 KM - 150 m2,LT : 90 m2 ( 6x15 )\n LB : 150 m2\n Lantai : 2\n KT / KM : 4 KT / 3 KM ( + 1 KT KM ART)\n Sertifikat : SHM\n View / Hadap : Utara
124,938313785,JUAL CEPAT! Metland Cakung Ujung Menteng Jakarta Timur,90.0,120.0,4 KT - 2 KM - 120 m2,"Rumah siap huni\n\nLingkungan tenang, nyaman dan asri\n\nRow jalan lebar 2 mobil \n\nTidak banjir \n\n\n\n\nLT 90 m2/LB 120 m2\n\nKT 4/KM 2\n\nAC 3 unit\n\nSHM \n\nStrategis dekat pintu tol cakung..."
132,927153229,Rumah Tinggi Dari Jalan Ada Lift Barang di BCS Kelapa Gading,120.0,1.0,4 KT - 2 KM - 1 m2,"JUAL CEPAT! Rumah Tinggi dari Jalan Ada Lift Barang di BCS Kelapa Gading, Nego Tipis, Bisa dibantu KPR sampai gooaall\n \n\n LT : 120 m2 (8x15)\n \n\n 2 Lantai\n KT : 4\n KM : 2\n PAM\n Ada Lift B..."



Sample p_alamat (some rows where p_alamat non-null but short/looks junk):


,ad_id,p_alamat,sublocality_name,city_name,province_name,description
6808,932752923,"strategis dan akses mudah, design modern dan elegan, fasilitas lengkap nyaman, harga terjangkau, bebas banjir",Cikarang Selatan,Bekasi Kab.,Jawa Barat,"Rumah modern yg dilengkapi dgn skylight dan high ceiling yg memastikan sirkulasi udara lebih lancar dan pencahayaan alami lebih maksimal\nFasilitas sekitar lengkap dan akses yg mudah, hanya 10 men..."
15322,938429601,"Strategis Cikini, Menteng.",Menteng,Jakarta Pusat,Jakarta D.K.I.,Apartemen Menteng Park\n\n\n\nTipe: 2 bedroom\n\n\n\nLuas: 58\n\n\n\nKondisi: Full Furnish\n\n\n\n\n\n\n\nFasilitas:\n\n\n\n1. Keamanan 24 jam\n\n\n\n2. Area sekitar restoran\n\n\n\n3. Supermarket...
23955,932183320,"strategis, keamanan 24 jam berada dijalan 2 jalur, dekat dengan pusat perbelanjaan Hypermart, BSD Plasa, Superindo, juga dekat pintu tol Jelupang[3menit]. Unit hadap Barat, air PAM, sudah baja rin...",Serpong Utara,Tangerang Selatan Kota,Banten,"Lokasi strategis, keamanan 24 jam berada dijalan 2 jalur, dekat dengan pusat perbelanjaan Hypermart, BSD Plasa, Superindo, juga dekat pintu tol Jelupang[3menit]. Unit hadap Barat, air PAM, sudah b..."
5307,930860618,sangat strategis dekat dengan - Mcd Raden Inten- Tol Becakayu- SMPN 194- Komplek Ikip,Duren Sawit,Jakarta Timur,Jakarta D.K.I.,Rumah dijual di Duren Sawit JaktimBangunan Indent bisa request denah dalam rumahnyaHarga 1.850 milyar negoFree AC 1 PKLt 118m Lb 160m Kamar tidur 4 Kamar mandi 3 Listrik 2200 wattAkses jalan 2 mob...
12607,926492477,Aman Tenang dan Bebas Banjir,Cinere,Depok Kota,Jawa Barat,Dijual Rumah Murah Dalam Cluster Lenteng Agung Jakarta Selatan\n \n Selling Point :\n - One Gate Sistem\n - Akses Jalan 2 Mobil\n - Lokasi Aman Tenang dan Bebas Banjir\n - Akses Dekat ke Stasiun U...
5415,939951440,strategis,Jatiasih,Bekasi Kota,Jawa Barat,"Rumah 2Lantai Modern Exsclusif Termurah Sejatiasih, Desain Minimalis dan Harga Terjangkau Di Jatiasih Akses 2 Mobil dan Bebas Banjir\n \n Harga Mulai 625 Jt\n Booking Fee 3 Jt \n DP 0% \n Free sem..."
24638,936827569,"Premium: Cluster Boston, Gading Serpong Tangerang",Gading Serpong,Tangerang Kota,Banten,"Dijual Rumah Siap Huni Full Furnish di Cluster Boston Gading Serpong\n Lokasi Premium: Cluster Boston, Gading Serpong Tangerang\n Tipe Hook: Posisi strategis di pojokan, pencahayaan dan sirkulasi..."
8347,934972199,Sangat Strategis.,Cipayung,Depok Kota,Jawa Barat,Promo Rumah Terlaris Di Bojong Gede.\n Rumah Mewah Minimalis Modern Bebas Banjir Lokasi Sangat Strategis.\n Jalur Mobil Di Lalui Angkot menuju stasiun Bojong Gede.\n Pembayaran Hanya Bisa Cash Ker...
15413,938407284,"strategis, pinggir jalan raya, bebas banjir, akses jalan lebar",Palmerah,Jakarta Barat,Jakarta D.K.I.,"· Cash only (tunai)\n\n· Dijual melalui proses lelang Negara\n\n· Aset kredit macet, harga di bawah pasar\n\n\n\n\nRumah mewah, lokasi strategis, pinggir jalan raya, bebas banjir, akses jalan leba..."
793,936119452,unggulan:,Tambun Utara,Bekasi Kab.,Jawa Barat,"Buat kamu yang pengen rumah nyaman sekaligus peluang usaha, Rumah + Ruko 2 pintu di Pondok Ungu Permai, Bekasi ini jawabannya.\n Nggak cuma tempat tinggal, tapi juga siap jadi mesin bisnis kamu.\n..."


# Fungsi Ekstraksi "floor" yang Lebih Ketat dan Normalisasi

In [44]:
import re
def extract_floor_from_text(text):
    """Ekstraksi lantai: cari angka yang dekat kata lantai/lt/floor.
       Kembalikan int atau None.
    """
    if not text or not isinstance(text, str):
        return None
    txt = text.lower()
    # patterns to catch e.g. "2 lantai", "lt. 2", "lt 2", "floor 2", "2 lt", "2 lantai (hook?)", "2 1/2 lantai"
    # handle 1/2 fractions -> round down to int (or drop, depending policy). We'll round to nearest int.
    # pattern for "2 1/2" or "2½"
    frac_pat = re.search(r'(\d+)\s*(?:1\/2|½)', txt)
    if frac_pat:
        try:
            val = int(frac_pat.group(1))
            return val  # keep integer portion (or you can add +1)
        except:
            pass
    # strict patterns: number adjacent to lantai/lt/floor
    m = re.search(r'(?:lantai|lt|lt\.|floor)[\s:\-]*([0-9]{1,3})\b', txt)
    if m:
        try:
            return int(m.group(1))
        except:
            pass
    # variants like "lt 2" or "lt2" or "2 lt"
    m = re.search(r'\b([0-9]{1,3})\s*(?:lantai|lt|lt\.|floor)\b', txt)
    if m:
        try:
            return int(m.group(1))
        except:
            pass
    # fallback: look for "lantai" followed by words then number (rare)
    m = re.search(r'lantai[^\d]{0,10}([0-9]{1,3})', txt)
    if m:
        try:
            return int(m.group(1))
        except:
            pass
    return None

def normalize_floor_value(v):
    """Normalize numeric floor: if v > 50 or negative -> treat as NaN (suspect).
       You can adjust the threshold (50) if you expect high-rise building floors.
    """
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        iv = int(float(v))
        if iv < 0:
            return None
        # Threshold: > 50 likely mis-extraction for our dataset (houses/apartments)
        if iv > 50:
            return None
        return iv
    except:
        return None

# Normalisasi "_params_dict"

In [45]:
import json
from ast import literal_eval

def ensure_dict(x):
    # If already dict, return as is
    if isinstance(x, dict):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    # If it's already a dict-like string (JSON), try parse
    if isinstance(x, str):
        s = x.strip()
        # try json loads
        try:
            parsed = json.loads(s)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass
        # try literal_eval (handles python dict repr)
        try:
            parsed = literal_eval(s)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass
        # fallback: try to extract simple "k":"v" pairs with regex
        try:
            # very small heuristic: find "key": "value" pairs
            kv = {}
            import re
            for m in re.finditer(r'"?([A-Za-z0-9_\-]+)"?\s*:\s*"([^"]+)"', s):
                kv[m.group(1)] = m.group(2)
            return kv
        except Exception:
            return {}
    # if it's a list/tuple -> try convert to dict-ish? -> fallback empty
    return {}

# apply normalization in-place (this can be a bit slow on many rows)
if '_params_dict' in df.columns:
    df['_params_dict'] = df['_params_dict'].apply(ensure_dict)
else:
    # if not present, try create from raw_parameters if possible
    def build_params(x):
        try:
            return json.loads(x) if isinstance(x, str) else {}
        except:
            return {}
    df['_params_dict'] = df.get('raw_parameters', pd.Series([None]*len(df))).apply(build_params)

# quick sanity check: ensure all are dicts now
non_dict_count = df['_params_dict'].apply(lambda v: not isinstance(v, dict)).sum()
print("Non-dict entries in _params_dict after normalization:", non_dict_count)
# show sample
df['_params_dict'].head(5)

Non-dict entries in _params_dict after normalization: 0


0    {'external_source_url': 'https://www.lamudi.co.id/properti/41032-73-cc711e14d66c-28db-198f4d0-ba29-7395?i=AI-_PlE', 'external_source_label': 'https://www.lamudi.co.id/properti/41032-73-cc711e14d66...
1    {'external_source_url': 'https://www.lamudi.co.id/properti/41032-73-cc64aaa12730-b7a7-19845f0-aadc-78fd?i=S2g08w', 'external_source_label': 'https://www.lamudi.co.id/properti/41032-73-cc64aaa12730...
2    {'external_source_label': 'https://www.lamudi.co.id/proyek-baru/sl-samesta-east-point/samesta-east-point-tipe-2-bedroom-172198713672/', 'external_source_url': 'https://www.lamudi.co.id/proyek-baru...
3    {'external_source_url': 'https://www.lamudi.co.id/properti/41032-73-abd2260c8be6-be2d-1999b86-bc43-722b?i=AKdYKvk', 'external_source_label': 'https://www.lamudi.co.id/properti/41032-73-abd2260c8be...
4    {'external_source_url': 'https://www.lamudi.co.id/properti/41032-73-d3c1af6b3a0d-38d7-1999b13-ae1d-7d02?i=SGikSg', 'external_source_label': 'https://www.lamudi.co.id/properti/

# Terapkan Ekstraksi Ulang dan Sanity Check ke Seluruh Dataset

In [46]:
import re
import numpy as np

def extract_floor_from_text(text):
    if not text or not isinstance(text, str):
        return None
    txt = text.lower()
    # fractions like "2 1/2" or "2½"
    frac_pat = re.search(r'(\d+)\s*(?:1\/2|½)', txt)
    if frac_pat:
        try:
            val = int(frac_pat.group(1))
            return val
        except:
            pass
    # strict patterns near lantai/lt/floor
    m = re.search(r'(?:lantai|lt|lt\.|floor)[\s:\-]*([0-9]{1,3})\b', txt)
    if m:
        try:
            return int(m.group(1))
        except:
            pass
    m = re.search(r'\b([0-9]{1,3})\s*(?:lantai|lt|lt\.|floor)\b', txt)
    if m:
        try:
            return int(m.group(1))
        except:
            pass
    m = re.search(r'lantai[^\d]{0,10}([0-9]{1,3})', txt)
    if m:
        try:
            return int(m.group(1))
        except:
            pass
    return None

def normalize_floor_value(v, max_floor=50):
    if v is None:
        return None
    try:
        iv = int(float(v))
    except:
        return None
    if iv < 0:
        return None
    if iv > max_floor:
        return None
    return iv

def recompute_floor_safe(row):
    # row is a Series
    # 1) try original explicit 'floor' if reasonable
    cur = row.get('floor')
    try:
        orig = int(float(cur)) if (cur is not None and str(cur).strip()!='' and not pd.isna(cur)) else None
    except:
        orig = None
    if orig is not None:
        nf = normalize_floor_value(orig)
        if nf is not None:
            return nf

    # 2) try _params_dict if present and is dict
    params = row.get('_params_dict') if '_params_dict' in row.index else None
    if isinstance(params, str):
        # defensive: parse string
        try:
            params = json.loads(params)
        except:
            try:
                from ast import literal_eval
                params = literal_eval(params)
            except:
                params = {}
    if not isinstance(params, dict):
        params = {}

    for k in ('p_floor','floor','p_floor_label','p_floor_number'):
        if k in params and params[k] not in (None, "", [], {}):
            # try to extract number
            try:
                s = str(params[k])
                num = re.search(r'([0-9]{1,3})', s)
                if num:
                    nf = normalize_floor_value(int(num.group(1)))
                    if nf is not None:
                        return nf
            except:
                pass

    # 3) try main_info then description then title (in that order)
    for field in ('main_info','description','title'):
        text = row.get(field)
        v = extract_floor_from_text(str(text) if pd.notna(text) else "")
        if v is not None:
            nv = normalize_floor_value(v)
            if nv is not None:
                return nv

    # not found
    return None

# Apply recompute (this may take some time)
df['floor_clean'] = df.apply(recompute_floor_safe, axis=1)

print("Original non-null floor:", df['floor'].notnull().sum())
print("New floor_clean non-null:", df['floor_clean'].notnull().sum())

# Optionally replace original floor with floor_clean for rows where floor_clean not null
df['floor'] = df['floor_clean'].combine_first(df['floor'])
df.drop(columns=['floor_clean'], inplace=True)

Original non-null floor: 18877
New floor_clean non-null: 16509


# Bersihkan "p_alamat" dan Buat Kolom "full_address"

In [47]:
# fungsi pembersihan p_alamat
import re
def clean_p_alamat_raw(txt, keep_min_words=2):
    if not txt or not isinstance(txt, str):
        return None
    s = txt.strip()
    # remove control chars & collapse whitespace
    s = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    # remove obvious junk phrases
    junk_patterns = [
        r'\bpromo\b', r'\bdijual\b', r'\bjual\b', r'\bhub\b', r'\bwa\b', r'\bcontact\b',
        r'\bcp\b', r'\binfo\b', r'\bcek\b', r'\bharga\b', r'\brp\b', r'\bnego\b'
    ]
    for jp in junk_patterns:
        s = re.sub(jp, '', s, flags=re.I).strip()
    # if s looks like single short token (e.g. just 'Bekasi'), keep it if >= keep_min_words words
    if len(s.split()) < keep_min_words:
        # return None if too short and not informative
        return s if len(s) >= 3 else None
    return s

# apply cleaning
df['p_alamat_clean'] = df['p_alamat'].apply(clean_p_alamat_raw)

# build full_address by combining p_alamat_clean + sublocality + city + province
def build_full_address(row):
    parts = []
    if row.get('p_alamat_clean'):
        parts.append(row['p_alamat_clean'])
    # prefer sublocality if available (more granular)
    for k in ('sublocality_name','city_name','province_name'):
        v = row.get(k)
        if v and str(v).strip() and str(v).lower() not in ('nan','none'):
            parts.append(str(v).strip())
    if parts:
        # join unique preserving order
        seen = set(); final = []
        for p in parts:
            if p not in seen:
                final.append(p); seen.add(p)
        addr = ", ".join(final) + ", Indonesia"
        return addr
    return None

df['full_address'] = df.apply(build_full_address, axis=1)

# show before/after counts
print("p_alamat non-null (orig):", df['p_alamat'].notnull().sum())
print("p_alamat_clean non-null:", df['p_alamat_clean'].notnull().sum())
print("full_address non-null:", df['full_address'].notnull().sum())

# sample examples
display(df[['ad_id','p_alamat','p_alamat_clean','sublocality_name','city_name','province_name','full_address']].sample(n=10, random_state=42))

p_alamat non-null (orig): 12300
p_alamat_clean non-null: 12102
full_address non-null: 25384


,ad_id,p_alamat,p_alamat_clean,sublocality_name,city_name,province_name,full_address
24770,932087734,NaN,None,Gading Serpong,Tangerang Kota,Banten,"Gading Serpong, Tangerang Kota, Banten, Indonesia"
2766,929392326,NaN,None,Cikarang Timur,Bekasi Kab.,Jawa Barat,"Cikarang Timur, Bekasi Kab., Jawa Barat, Indonesia"
5345,939926736,Dalam Komplek Bintara jaya Dekat Kalimalang Sumber Arta,Dalam Komplek Bintara jaya Dekat Kalimalang Sumber Arta,Bekasi Barat,Bekasi Kota,Jawa Barat,"Dalam Komplek Bintara jaya Dekat Kalimalang Sumber Arta, Bekasi Barat, Bekasi Kota, Jawa Barat, Indonesia"
22425,937435510,.,None,Sepatan,Tangerang Kab.,Banten,"Sepatan, Tangerang Kab., Banten, Indonesia"
12804,938421595,"setrategis,aman nyaman & tidak banjir..","setrategis,aman nyaman & tidak banjir..",Limo,Depok Kota,Jawa Barat,"setrategis,aman nyaman & tidak banjir.., Limo, Depok Kota, Jawa Barat, Indonesia"
11766,928526457,. Kp carang pulang Rt 01/04 Desa Cikarawang Kec Dramaga Bogor belakang IPB ke IPB sekitaran 5-10 pake motor.,. Kp carang pulang Rt 01/04 Desa Cikarawang Kec Dramaga Bogor belakang IPB ke IPB sekitaran 5-10 pake motor.,Dramaga,Bogor Kab.,Jawa Barat,". Kp carang pulang Rt 01/04 Desa Cikarawang Kec Dramaga Bogor belakang IPB ke IPB sekitaran 5-10 pake motor., Dramaga, Bogor Kab., Jawa Barat, Indonesia"
7792,939156683,"Indraprasta, jl. dewi kunthi","Indraprasta, jl. dewi kunthi",Bogor Utara - Kota,Bogor Kota,Jawa Barat,"Indraprasta, jl. dewi kunthi, Bogor Utara - Kota, Bogor Kota, Jawa Barat, Indonesia"
8288,935566848,NaN,None,Cibinong,Bogor Kab.,Jawa Barat,"Cibinong, Bogor Kab., Jawa Barat, Indonesia"
24807,934600162,NaN,None,Pondok Aren,Tangerang Selatan Kota,Banten,"Pondok Aren, Tangerang Selatan Kota, Banten, Indonesia"
14653,939894939,NaN,None,Cengkareng,Jakarta Barat,Jakarta D.K.I.,"Cengkareng, Jakarta Barat, Jakarta D.K.I., Indonesia"


# Replace Final Columns, Drop Helpers, Simpan ke Excel

In [48]:
# keep p_alamat_clean as p_alamat or keep both
df['p_alamat'] = df['p_alamat_clean']

# drop helper cols
for c in ['floor_num','floor_clean','_params_dict','_main_bed','_main_bath','_main_area','p_alamat_clean']:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

# Save to Excel (sanitasi minimal again)
def sanitize_text_simple(s):
    if s is None: return None
    s = str(s)
    s = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    if len(s) > 32000:
        s = s[:32000-3] + "..."
    return s

for c in df.columns:
    # convert lists/dicts => json strings (should already be strings)
    df[c] = df[c].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x,(dict,list,tuple)) else x)
    df[c] = df[c].apply(sanitize_text_simple)

OUT_XLSX = "processed_dataset3.xlsx"
df.to_excel(OUT_XLSX, index=False)
print("Saved cleaned Excel:", OUT_XLSX)

Saved cleaned Excel: processed_dataset3.xlsx
